# Training on INCLUDE_50 Dataset

## Preprocessing Data

In [67]:
import pandas as pd
import numpy as np
import os

from tqdm.notebook import tqdm

In [6]:
train_data = pd.read_csv('Dataset/train.csv')
test_data = pd.read_csv('Dataset/test.csv')

train_data.sample(5)

,parent_label,label,video_path,include_50
791,Means_of_Transportation,10. Plane,Means_of_Transportation/10. Plane/MVI_3115.mp4,False
2361,Adjectives,33. dead,Adjectives/33. dead/MVI_9702.mp4,False
2030,Means_of_Transportation,10. Plane,Means_of_Transportation/10. Plane/MVI_3171.mp4,False
1707,Jobs,95. Author,Jobs/95. Author/MVI_4506.mp4,False
2969,Days_and_Time,66. Sunday,Days_and_Time/66. Sunday/MVI_5443.mp4,False


In [63]:
# labels = train_data['label']
# train_data_path = train_data['vedio_path']


labels = []
train_path_I50 = []

for i in range(len(train_data)):
    if train_data['include_50'][i] == True:
        labels.append(train_data['label'][i])
        train_path_I50.append("Dataset\\" + train_data['video_path'][i])
        
labels = pd.Series(labels).unique()
labels = pd.Series(labels).to_list()

train_path_I50 = pd.Series(train_path_I50)

# train_path_I50.sample(5), labels.sample(5)
labels[:5]

['1. loud', '19. House', '83. big large', '91. Priest', '23. Court']

In [64]:
label_map = dict()

for i in range(len(labels)):
    split = labels[i].split(" ")
        
    label = split[1:]
    label = " ".join(label)
    
    label_map[label] = int(split[0].split(".")[0])

labels = list(label_map.keys())
label_map    

{'loud': 1,
 'House': 19,
 'big large': 83,
 'Priest': 91,
 'Court': 23,
 'train ticket': 16,
 'it': 44,
 'Shoes': 44,
 'Dog': 1,
 'Bank': 35,
 'Thank you': 55,
 'Election': 14,
 'Cow': 5,
 'Window': 28,
 'quiet': 2,
 'dry': 97,
 'long': 78,
 'Hello': 48,
 'Bird': 4,
 'Hat': 37,
 'Black': 54,
 'short': 79,
 'White': 55,
 'Fan': 53,
 'new': 91,
 'Store or Shop': 28,
 'Monday': 67,
 'Death': 2,
 'Cell phone': 54,
 'you (plural)': 46,
 'T-Shirt': 42,
 'Girl': 78,
 'Father': 61,
 'Red': 47,
 'hot': 87,
 'Fall': 64,
 'I': 40,
 'Time': 86,
 'Car': 11,
 'Good Morning': 51,
 'Summer': 61,
 'Paint': 40,
 'Teacher': 84,
 'Brother': 66,
 'good': 94,
 'happy': 3,
 'Boy': 77,
 'small little': 84,
 'Pen': 34,
 'Year': 78}

In [77]:
video_frames = []

for label in tqdm(labels):
    label_videos = os.listdir("MP_data/"+label)
    window = []
    # print(label_videos)
    for video in label_videos:
        res = np.load("MP_data/" + f"{label}/" + video)
        window.append(res)
    video_frames.append(window)

        

  0%|          | 0/50 [00:00<?, ?it/s]

In [78]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Assuming each frame should be a 1D array of features
max_features = max(max(len(frame) for frame in video) for video in video_frames)

# Pad or truncate each frame to have the same number of features
normalized_frames = [[np.pad(frame, (0, max_features - len(frame)), 'constant') 
                      if len(frame) < max_features else frame[:max_features] 
                      for frame in video] 
                     for video in video_frames]

# Now pad sequences
max_length = max(len(seq) for seq in normalized_frames)
padded_video_frames = pad_sequences(normalized_frames, maxlen=max_length, dtype='float32', padding='post', truncating='post')

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (13, 154) + inhomogeneous part.

In [76]:
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.utils import to_categorical

# Pad sequences to ensure they have the same length
max_length = max(len(seq) for seq in video_frames)
padded_video_frames = pad_sequences(video_frames, maxlen=max_length, dtype='float32', padding='post', truncating='post')

X = np.array(padded_video_frames)
# X = np.array(video_frames)
y = to_categorical(labels).astype(int)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (13,) + inhomogeneous part.